# 区间和
## 问题
给定一个整数数组 nums，以及若干次区间查询。每次查询给出下标 left 和 right，计算左闭右闭区间 [left, right] 内所有元素的和

[卡码网题目链接](https://kamacoder.com/problempage.php?pid=1070)

数组下标从 0 开始，查询满足 0 <= left <= right < n，查询过程中数组元素不发生变化

### 输入与输出
第一行是数组长度 n，接下来 n 行每行一个整数。之后每行给出两个区间下标，查询一直读到文件结束，没有单独给出查询次数

每次查询输出一行区间和

输入示例：

```text
5
1
2
3
4
5
0 1
1 3
```

输出示例：

```text
3
9
```

解释：nums = [1,2,3,4,5]，区间 [0,1] 的和为 1 + 2 = 3，区间 [1,3] 的和为 2 + 3 + 4 = 9

## 思路
如果只查询一次，可以直接遍历指定区间，累加其中的元素

但查询次数很多时，不同区间可能包含大量相同元素，每次从头累加会产生重复计算

前缀和的做法是先保存“数组前若干个元素的和”，之后用两个前缀和相减，就能得到任意区间的和

与滑动窗口依赖窗口扩张和收缩不同，本题的查询区间已经给定，而且元素可以为负数。前缀和直接利用加减运算，不要求数组元素为正数

## 解法
### 暴力求解
对每次查询，遍历下标 left 到 right，将元素依次累加

因为查询区间包含 right，遍历范围要写成 range(left, right + 1)

为了方便在 notebook 中调用，下面的方法接收数组 nums 和查询列表 queries，并返回每次查询的结果；题目的标准输入输出写法放在后面


In [ ]:
class solution:
    def rangeSum(
        self, nums: list[int], queries: list[tuple[int, int]]
    ) -> list[int]:
        result = []

        for left, right in queries:
            total = 0

            for index in range(left, right + 1):
                total += nums[index]

            result.append(total)

        return result


设数组长度为 n，查询次数为 q

时间复杂度：单次查询为 $O(right - left + 1)$，最坏为 $O(n)$；q 次查询最坏为 $O(qn)$

空间复杂度：返回结果需要 $O(q)$ 空间；如果不计返回结果，额外空间为 $O(1)$

写成 sum(nums[left:right + 1]) 也能求和，但仍需要遍历区间，而且列表切片会创建新列表，不能将它当作 $O(1)$ 的查询


### 前缀和
创建长度为 n + 1 的数组 prefix，定义：

**prefix[i] 表示 nums 的前 i 个元素之和，也就是左闭右开区间 [0, i) 的和**

注意：i 表示元素个数，不是最后一个元素的下标，因此 prefix[i] 不包含 nums[i]

prefix[0] = 0，表示前 0 个元素的和。之后逐个累加：

```python
prefix[i + 1] = prefix[i] + nums[i]
```

以 nums = [1,2,3,4,5] 为例：

| i | prefix[i] 对应的元素 | prefix[i] |
| --- | --- | --- |
| 0 | 没有元素 | 0 |
| 1 | [1] | 1 |
| 2 | [1,2] | 3 |
| 3 | [1,2,3] | 6 |
| 4 | [1,2,3,4] | 10 |
| 5 | [1,2,3,4,5] | 15 |

查询左闭右闭区间 [left, right] 时：
- prefix[right + 1] 是从下标 0 到 right 的和，包含 right
- prefix[left] 是从下标 0 到 left - 1 的和，不包含 left
- 两者相减，去掉 left 之前的元素，剩下的就是所求区间

因此：

$$
\text{区间和}(left, right) = prefix[right + 1] - prefix[left]
$$

例如查询 [1,3]：

```text
prefix[4] = 1 + 2 + 3 + 4 = 10
prefix[1] = 1
区间和 = 10 - 1 = 9
```

为什么要多放一个 prefix[0]？

如果区间从下标 0 开始，仍然可以使用同一个公式：prefix[right + 1] - prefix[0]，不需要单独判断 left == 0

这也避免了使用 prefix[left - 1] 时，left == 0 导致下标变成 -1 的问题。在 Python 中，-1 会取最后一个元素，通常不会报错，却会算出错误结果


In [ ]:
class solution:
    def rangeSum(
        self, nums: list[int], queries: list[tuple[int, int]]
    ) -> list[int]:
        prefix = [0] * (len(nums) + 1)

        for index in range(len(nums)):
            prefix[index + 1] = prefix[index] + nums[index]

        result = []

        for left, right in queries:
            result.append(prefix[right + 1] - prefix[left])

        return result


时间复杂度：构建前缀和需要 $O(n)$，构建后每次查询只做一次减法，为 $O(1)$；处理 q 次查询的总时间为 $O(n + q)$

空间复杂度：prefix 需要 $O(n)$ 空间，返回结果需要 $O(q)$ 空间，总共为 $O(n + q)$；如果不计返回结果，额外空间为 $O(n)$

例如调用 solution().rangeSum([1,2,3,4,5], [(0,1), (1,3)])，返回 [3,9]

### 易错点
- 先明确前缀和的定义：本笔记的 prefix[i] 是前 i 个元素的和，不包含 nums[i]
- prefix 的长度为 n + 1，prefix[0] 为 0，prefix[n] 才是整个数组的和
- 查询区间 [left, right] 包含两端，所以右侧使用 prefix[right + 1]
- left == right 时，查询的是单个元素，结果为 prefix[left + 1] - prefix[left]
- 前缀和只需在所有查询开始前构建一次，不要在每次查询时重新构建
- 当前写法适合数组不变、反复查询的场景；如果修改了 nums 的元素，原先构建的 prefix 就不能直接继续使用


### 标准输入输出写法
卡码网需要完整程序读取输入并打印结果，可以将下面的代码保存为 .py 文件或直接提交

这里在读取数组元素时就构建 prefix，不需要另外保存 nums；查询也逐行处理，不需要保存所有查询和结果

```python
import sys

n = int(sys.stdin.readline())
prefix = [0] * (n + 1)

for index in range(n):
    num = int(sys.stdin.readline())
    prefix[index + 1] = prefix[index] + num

for line in sys.stdin:
    if not line.strip():
        continue

    left, right = map(int, line.split())
    print(prefix[right + 1] - prefix[left])
```

for line in sys.stdin 会逐行读取剩余输入，直到文件结束；line.split() 将两个下标拆开，map(int, ...) 将它们转换成整数，left, right 接收这两个值

这个程序的总时间复杂度为 $O(n + q)$，空间复杂度为 $O(n)$
